# 02b — Feature extraction (text, image, knowledge)
MAS Lab 2 — Fake news detection (WCO + MOA)

Input: `finefake_clean.pkl` from notebook 02a (attached as input).
Output: one numeric vector per news item for each modality + PCA-reduced versions, saved as `.npy`.

| Modality | Model | Size |
|---|---|---|
| Text | `all-mpnet-base-v2` (sentence transformer) | 768 |
| Image | `clip-ViT-B-32` (CLIP image encoder) | 512 |
| Knowledge | `knowledge_embedding` (provided by FineFake) | checked below |

**Accelerator: GPU** · Internet: ON (to download the models)

## 0. Setup: GPU check, libraries, input data

In [1]:
!pip install -q sentence-transformers
import os, glob, json, zipfile
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'no GPU -> will be slow')

OUT = '/kaggle/working/features'
os.makedirs(OUT, exist_ok=True)

found = glob.glob('/kaggle/input/**/finefake_clean.pkl', recursive=True)
assert found, 'finefake_clean.pkl not found: attach the 02a_finefake_cleaning output (Add Input)'
IN_PKL = found[0]
df = pd.read_pickle(IN_PKL)
print('Loaded:', IN_PKL, df.shape)

Device: cuda | Tesla T4
Loaded: /kaggle/input/notebooks/khitersaranadjet/02a-finefake-cleaning/clean/finefake_clean.pkl (15298, 9)


### 0b. Image paths
In 02a the images were in `/kaggle/working/finefake/`. In this notebook they come from the 02a output, mounted under `/kaggle/input/...` (read-only).
We rewrite the paths. If the images are not in the input, we download them again.

In [2]:
OLD_BASE = '/kaggle/working/finefake'

INPUT_ROOT = os.path.dirname(os.path.dirname(IN_PKL))
img_dirs = [os.path.join(INPUT_ROOT, 'finefake', 'Image')]
img_dirs = [d for d in img_dirs if os.path.isdir(d)]

if img_dirs:
    NEW_BASE = os.path.dirname(img_dirs[0])
    print('Images found in input:', NEW_BASE)
else:
    !pip install -q gdown
    import gdown
    NEW_BASE = OLD_BASE
    archive = os.path.join(NEW_BASE, 'finefake_archive')
    os.makedirs(NEW_BASE, exist_ok=True)
    gdown.download(id='16D9ix7ZOisa4VVBznBTBcv1N7TA-jodH', output=archive, quiet=False)
    with zipfile.ZipFile(archive) as z:
        z.extractall(NEW_BASE)
    os.remove(archive)
    print('Images downloaded to:', NEW_BASE)

df['image_full'] = df['image_full'].apply(lambda p: p.replace(OLD_BASE, NEW_BASE, 1) if isinstance(p, str) else None)
print('Example path:', df.loc[df['has_image'], 'image_full'].iloc[0])
print('Exists:', os.path.exists(df.loc[df['has_image'], 'image_full'].iloc[0]))

Images found in input: /kaggle/input/notebooks/khitersaranadjet/02a-finefake-cleaning/finefake
Example path: /kaggle/input/notebooks/khitersaranadjet/02a-finefake-cleaning/finefake/Image/cdc_gov/13.jpeg
Exists: True


## 1. Text embeddings — `all-mpnet-base-v2`
Each cleaned text -> tokens -> transformer -> one 768-number vector.
Texts longer than 384 tokens (~280–300 words) are truncated (title + beginning are kept).

In [3]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer('all-mpnet-base-v2', device=DEVICE)
text_model.max_seq_length = 384

n_tokens = [len(t) for t in text_model.tokenizer(df['text_clean'].tolist(), add_special_tokens=True)['input_ids']]
print(f'Texts truncated (> 384 tokens): {np.mean(np.array(n_tokens) > 384):.1%}')

X_text = text_model.encode(
    df['text_clean'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)
print('X_text:', X_text.shape)

Texts truncated (> 384 tokens): 17.8%


X_text: (15298, 768)


## 2. Image embeddings — CLIP `clip-ViT-B-32`
Each valid image -> resized 224×224 -> 49 patches -> vision transformer -> one 512-number vector.
Rows without a valid image get a vector of zeros (and `has_image = 0`).
Images that fail to load here (e.g. truncated files that passed the header check in 02a) are also set to *no image*.

In [4]:
img_model = SentenceTransformer('clip-ViT-B-32', device=DEVICE)

X_img = np.zeros((len(df), 512), dtype=np.float32)
has_image = df['has_image'].to_numpy().copy()

def load_image(path):
    try:
        im = Image.open(path)
        im.draft('RGB', (448, 448))
        im = im.convert('RGB')
        im.thumbnail((448, 448))
        return im
    except Exception:
        return None

idx_valid = np.where(has_image)[0]
BATCH = 128
n_failed = 0
for start in tqdm(range(0, len(idx_valid), BATCH)):
    rows = idx_valid[start:start + BATCH]
    imgs, ok_rows = [], []
    for r in rows:
        im = load_image(df['image_full'].iat[r])
        if im is None:
            has_image[r] = False
            n_failed += 1
        else:
            imgs.append(im); ok_rows.append(r)
    if imgs:
        X_img[ok_rows] = img_model.encode(imgs, batch_size=BATCH, convert_to_numpy=True,
                                          normalize_embeddings=True, show_progress_bar=False)

df['has_image'] = has_image
df.loc[~df['has_image'], 'image_full'] = None
print('Images that failed at full load:', n_failed)
print('X_img:', X_img.shape, '| rows with an image vector:', int(has_image.sum()))

Images that failed at full load: 0
X_img: (15298, 512) | rows with an image vector: 15163


## 3. Knowledge embedding (provided by FineFake)
Already numbers: we only convert it to a matrix and check its size and how many rows are empty (all zeros).

In [5]:
lengths = df['knowledge_embedding'].apply(lambda v: len(v) if v is not None else 0)
print('Vector lengths found:'); print(lengths.value_counts().head())

KG_DIM = int(lengths[lengths > 0].mode()[0])
def to_vec(v):
    v = np.asarray(v, dtype=np.float32) if v is not None else np.array([], dtype=np.float32)
    return v if len(v) == KG_DIM else np.zeros(KG_DIM, dtype=np.float32)

X_kg = np.stack(df['knowledge_embedding'].apply(to_vec).to_list())
zero_rows = (np.abs(X_kg).sum(axis=1) == 0)
print('X_kg:', X_kg.shape, f'| all-zero rows (no entity found): {zero_rows.mean():.1%}')
print('Zero rows by label:'); print(pd.Series(zero_rows).groupby(df['label'].to_numpy()).mean().round(3))

Vector lengths found:
knowledge_embedding
100    15298
Name: count, dtype: int64
X_kg: (15298, 100) | all-zero rows (no entity found): 20.8%
Zero rows by label:
0    0.256
1    0.155
dtype: float64


## 4. Sanity checks
Shapes aligned, no NaN, and a meaning check: the closest texts to a given text should be about the same thing.

In [6]:
for name, X in [('text', X_text), ('image', X_img), ('knowledge', X_kg)]:
    print(f'{name:9s} shape={X.shape}  NaN={np.isnan(X).sum()}  mean={X.mean():.4f}  std={X.std():.4f}')
assert len(X_text) == len(X_img) == len(X_kg) == len(df)

q = df.index[df['text_clean'].str.len().between(60, 200)][0]
sims = X_text @ X_text[q]
top = np.argsort(-sims)[1:4]
print('\nQUERY  :', df.at[q, 'text_clean'][:120], '| label', df.at[q, 'label'])
for i in top:
    print(f'{sims[i]:.3f} :', df.at[i, 'text_clean'][:120], '| label', df.at[i, 'label'])

text      shape=(15298, 768)  NaN=0  mean=-0.0001  std=0.0361
image     shape=(15298, 512)  NaN=0  mean=-0.0014  std=0.0440
knowledge shape=(15298, 100)  NaN=0  mean=-0.0005  std=0.0621

QUERY  : About Zika Virus DiseaseWhat You Need to Know Top 5 things everyone needs to know about Zika. | label 1
0.578 : Short Answers to Hard Questions About Zika VirusThe World Health Organization has declared an international health emerg | label 0
0.562 : Zika VirusFor Travelers Since 2018, no local mosquito-borne Zika virus transmission has been reported in the continental | label 1
0.519 : Zika Virus Spreads to New Areas - Region of the Americas, May 2015-January 2016The best way to prevent Zika virus infect | label 1


## 5. PCA — fitted on TRAIN rows only
Compress each block so the metaheuristic can train the classifier thousands of times quickly.
PCA learns its mean and directions from the train rows, then the same transformation is applied to all rows (no leakage from the test set).

In [7]:
from sklearn.decomposition import PCA

is_train = (df['split'] == 'train').to_numpy()
N_COMP = {'text': 128, 'image': 64, 'knowledge': min(32, KG_DIM)}
blocks = {'text': X_text, 'image': X_img, 'knowledge': X_kg}
reduced, pca_info = {}, {}

for name, X in blocks.items():
    pca = PCA(n_components=N_COMP[name], random_state=SEED)
    pca.fit(X[is_train])
    reduced[name] = pca.transform(X).astype(np.float32)
    cum = np.cumsum(pca.explained_variance_ratio_)
    pca_info[name] = {'n_components': N_COMP[name], 'explained_variance': round(float(cum[-1]), 4)}
    print(f'{name:9s} {X.shape[1]:4d} -> {N_COMP[name]:3d} dims | variance kept: {cum[-1]:.1%}'
          f' | first 10 comps: {cum[min(9, len(cum)-1)]:.1%}')

text       768 -> 128 dims | variance kept: 75.4% | first 10 comps: 20.7%
image      512 ->  64 dims | variance kept: 59.0% | first 10 comps: 29.0%
knowledge  100 ->  32 dims | variance kept: 55.1% | first 10 comps: 30.5%


## 6. Save
Full vectors (for possible later use) + PCA-reduced vectors (used in step 3) + labels/metadata. Row *i* of every file = row *i* of `meta.pkl`.

In [8]:
np.save(f'{OUT}/X_text.npy', X_text)
np.save(f'{OUT}/X_img.npy', X_img)
np.save(f'{OUT}/X_kg.npy', X_kg)
for name, Xr in reduced.items():
    np.save(f'{OUT}/X_{name}_pca.npy', Xr)

np.save(f'{OUT}/y.npy', df['label'].to_numpy().astype(np.int8))
np.save(f'{OUT}/is_train.npy', is_train)
np.save(f'{OUT}/has_image.npy', df['has_image'].to_numpy().astype(np.int8))

meta = df[['text_clean', 'image_full', 'has_image', 'label', 'fine_label', 'topic', 'platform', 'split']]
meta.to_pickle(f'{OUT}/meta.pkl')

with open(f'{OUT}/pca_info.json', 'w') as f:
    json.dump({'kg_dim': KG_DIM, 'pca': pca_info,
               'text_model': 'all-mpnet-base-v2', 'image_model': 'clip-ViT-B-32'}, f, indent=2)

for f in sorted(os.listdir(OUT)):
    print(f'{f:22s} {os.path.getsize(os.path.join(OUT, f)) / 1e6:7.1f} MB')

X_image_pca.npy            3.9 MB
X_img.npy                 31.3 MB
X_kg.npy                   6.1 MB
X_knowledge_pca.npy        2.0 MB
X_text.npy                47.0 MB
X_text_pca.npy             7.8 MB
has_image.npy              0.0 MB
is_train.npy               0.0 MB
meta.pkl                  23.4 MB
pca_info.json              0.0 MB
y.npy                      0.0 MB
